In [ ]:
import gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from gym.wrappers import FrameStack, GrayScaleObservation, ResizeObservation
from nes_py.wrappers import JoypadSpace
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
import time

# ===== 环境封装器 =====
def make_env():
    env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    env = GrayScaleObservation(env, keep_dim=True)
    env = ResizeObservation(env, shape=84)
    env = FrameStack(env, num_stack=4)
    return env

# ===== DQN 网络结构 =====
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        return self.net(x)

# ===== 经验回放池 =====
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        transitions = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*transitions)
        return np.array(state), action, reward, np.array(next_state), done

    def __len__(self):
        return len(self.buffer)

# ===== 动作选择策略 =====
def select_action(state, epsilon, policy_net, device):
    if random.random() > epsilon:
        with torch.no_grad():
            state = torch.tensor(np.array(state), dtype=torch.float32).unsqueeze(0).to(device)
            q_values = policy_net(state)
            return q_values.argmax(1).item()
    else:
        return random.randrange(len(SIMPLE_MOVEMENT))

# ===== 超参数设置 =====
BATCH_SIZE = 32
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.02
EPS_DECAY = 1000000
TARGET_UPDATE = 1000
LEARNING_RATE = 1e-4
MEMORY_SIZE = 100000
NUM_FRAMES = 2000000
SAVE_PATH = "dqn_mario.pth"

# ===== 初始化 =====
env = make_env()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy_net = DQN((4, 84, 84), len(SIMPLE_MOVEMENT)).to(device)
target_net = DQN((4, 84, 84), len(SIMPLE_MOVEMENT)).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LEARNING_RATE)
replay_buffer = ReplayBuffer(MEMORY_SIZE)
steps_done = 0
state = env.reset()
episode_reward = 0

# ===== 主训练循环 =====
for frame_idx in range(1, NUM_FRAMES + 1):
    epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * steps_done / EPS_DECAY)
    action = select_action(state, epsilon, policy_net, device)

    next_state, reward, done, info = env.step(action)
    replay_buffer.push(state, action, reward, next_state, done)
    state = next_state
    episode_reward += reward
    steps_done += 1

    if done:
        print(f"[Frame {frame_idx}] Episode Reward: {episode_reward}")
        state = env.reset()
        episode_reward = 0

    if len(replay_buffer) > BATCH_SIZE:
        s, a, r, s_, d = replay_buffer.sample(BATCH_SIZE)

        s = torch.tensor(np.array(s), dtype=torch.float32).to(device)
        a = torch.tensor(a).unsqueeze(1).to(device)
        r = torch.tensor(r, dtype=torch.float32).unsqueeze(1).to(device)
        s_ = torch.tensor(np.array(s_), dtype=torch.float32).to(device)
        d = torch.tensor(d, dtype=torch.float32).unsqueeze(1).to(device)

        q_values = policy_net(s).gather(1, a)
        next_q_values = target_net(s_).max(1)[0].detach().unsqueeze(1)
        expected_q_values = r + GAMMA * next_q_values * (1 - d)

        loss = nn.MSELoss()(q_values, expected_q_values)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if frame_idx % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if frame_idx % 10000 == 0:
        torch.save(policy_net.state_dict(), SAVE_PATH)
        print(f"✅ Saved model at frame {frame_idx}")

# ===== 保存最终模型 =====
torch.save(policy_net.state_dict(), SAVE_PATH)
print("🎉 Training complete! Model saved.")


: 